In [2]:
"""
Script 8 — Oligodendrocyte Marker Analysis
============================================
Generates Figures 21-23 for the results section:
  Figure 21 — Bar plot: oligodendrocyte marker expression per region
  Figure 22 — Heatmap: individual gene expression per region
  Figure 23 — Scatter: mature oligodendrocyte markers vs cholesterol synthesis

Dependencies: matplotlib, pandas, numpy, scipy, statsmodels
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')

# ── Style settings ────────────────────────────────────────────────────────────
plt.rcParams['font.family']       = 'Arial'
plt.rcParams['axes.facecolor']    = 'white'
plt.rcParams['figure.facecolor']  = 'white'
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

COLOR_MICROGLIA = '#6D597A'   
COLOR_LIPID     = '#E5989B'   
COLOR_AXONAL    = '#588157'   
COLOR_MITO      = '#84A59D'   
COLOR_ALS       = '#8C6275'   
COLOR_CTRL      = '#688A7A'   

COLOR_LCT     = '#3D5A80'
COLOR_DORSAL  = '#778D45'
COLOR_VENTRAL = '#CD7F6D'

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_PATH    = "/Users/annemijnrieff/Documents/Internship-data/Programming-data/Images-data/Region_clusters/Figures-plots2/"
DORSAL_PATH  = BASE_PATH + "LCT_vs_dorsal_ventral/Dorsal/"
VENTRAL_PATH = BASE_PATH + "LCT_vs_dorsal_ventral/Ventral/"
OUTPUT_PATH  = "/Users/annemijnrieff/Documents/Internship-data/Programming-data/Images-data/Region_clusters/Final_figures/Oligodendorcytes/"
HEALTHY_DORSAL  = "/Users/annemijnrieff/Documents/Internship-data/Programming-data/Images-data/Region_clusters/Healthy_hexpaint/Dorsal/"
ALS_DORSAL      = "/Users/annemijnrieff/Documents/Internship-data/Programming-data/Images-data/Region_clusters/Disease_hexpaint/Dorsal/"
HEALTHY_VENTRAL = "/Users/annemijnrieff/Documents/Internship-data/Programming-data/Images-data/Region_clusters/Healthy_hexpaint/Ventral/"
ALS_VENTRAL     = "/Users/annemijnrieff/Documents/Internship-data/Programming-data/Images-data/Region_clusters/Disease_hexpaint/Ventral/"
os.makedirs(OUTPUT_PATH, exist_ok=True)

WM_CLUSTERS = [4, 5, 6, 7]

# ── Oligodendrocyte gene sets ─────────────────────────────────────────────────
MATURE_OLIGO = ['ERMN', 'MAG', 'CLDN11', 'PLP1', 'MYRF', 'MOG', 'MOBP', 'MBP']
OPC_MARKERS  = ['OLIG1', 'OLIG2', 'SOX10', 'PDGFRA']
CHOL_SYNTH   = ['DHCR7', 'CYP51A1', 'SQLE', 'FDFT1', 'HMGCR']
ALL_OLIGO    = MATURE_OLIGO + OPC_MARKERS + CHOL_SYNTH

OLIGO_CATEGORIES = {
    'Mature oligodendrocytes': MATURE_OLIGO,
    'OPC markers':             OPC_MARKERS,
    'Oligo cholesterol synthesis': CHOL_SYNTH,
}

OLIGO_CAT_COLOR = {
    'Mature oligodendrocytes':     COLOR_AXONAL,
    'OPC markers':                 COLOR_MICROGLIA,
    'Oligo cholesterol synthesis': COLOR_LIPID,
}


# ════════════════════════════════════════════════════════════════════════════════
# DATA LOADING & DEG ANALYSIS
# ════════════════════════════════════════════════════════════════════════════════

def load_and_run_deg(healthy_path, als_path, output_csv, region_name):
    """Load or compute pseudo-bulk DEG analysis for a region."""
    if os.path.exists(output_csv):
        print(f"  {region_name}: loading existing DEG results")
        return pd.read_csv(output_csv, index_col=0)

    print(f"\n── {region_name}: loading raw data ─────────────────────────────")
    file_list = []
    groups    = {}

    for f in os.listdir(healthy_path):
        if f.endswith('.gz'):
            sid = f.split('_selection')[0]
            file_list.append((os.path.join(healthy_path, f), sid, 'Control'))
            groups[sid] = 'Control'
    for f in os.listdir(als_path):
        if f.endswith('.gz'):
            sid = f.split('_selection')[0]
            file_list.append((os.path.join(als_path, f), sid, 'ALS'))
            groups[sid] = 'ALS'

    pseudo_bulk = {}
    for full_path, sid, group in file_list:
        df = pd.read_csv(full_path, compression='gzip')
        df = df[df['L1_region_cluster'].isin(WM_CLUSTERS)]
        pseudo_bulk[sid] = df.groupby('geneName')['MIDCount'].sum()

    count_matrix  = pd.DataFrame(pseudo_bulk).fillna(0).astype(int)
    sample_groups = pd.Series(groups)
    als_samples   = sample_groups[sample_groups == 'ALS'].index.tolist()
    ctrl_samples  = sample_groups[sample_groups == 'Control'].index.tolist()

    lib_sizes   = count_matrix.sum(axis=0)
    median_lib  = lib_sizes.median()
    norm_matrix = count_matrix.divide(lib_sizes, axis=1) * median_lib

    results = []
    for gene in count_matrix.index:
        als_vals  = norm_matrix.loc[gene, als_samples].values
        ctrl_vals = norm_matrix.loc[gene, ctrl_samples].values
        if als_vals.mean() == 0 and ctrl_vals.mean() == 0: continue
        mean_als     = als_vals.mean() + 0.1
        mean_control = ctrl_vals.mean() + 0.1
        log2fc    = np.log2(mean_als / mean_control)
        baseMean  = (als_vals.mean() + ctrl_vals.mean()) / 2
        try:
            _, pvalue = mannwhitneyu(als_vals, ctrl_vals, alternative='two-sided')
        except:
            pvalue = 1.0
        results.append({'geneName': gene, 'log2FoldChange': log2fc,
                        'pvalue': pvalue, 'baseMean': baseMean})

    results_df = pd.DataFrame(results).set_index('geneName')
    _, padj, _, _ = multipletests(results_df['pvalue'], method='fdr_bh')
    results_df['padj'] = padj
    results_df.to_csv(output_csv)
    return results_df


def compute_region_summary(deg_df, region_name):
    """Compute mean log2FC and significance per oligodendrocyte category."""
    summary = {}
    for cat, genes in OLIGO_CATEGORIES.items():
        found = deg_df[deg_df.index.isin(genes)]
        if len(found) == 0: continue
        mean_lfc = found['log2FoldChange'].mean()
        _, p_val = stats.ttest_1samp(found['log2FoldChange'], 0)
        summary[cat] = {
            'mean_lfc': mean_lfc,
            'sem':      found['log2FoldChange'].sem(),
            'p':        p_val,
            'n_sig':    (found['padj'] < 0.05).sum(),
        }
    return summary


def save(fig, name, dpi=300):
    path = os.path.join(OUTPUT_PATH, name)
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"  Saved: {name}")


# ════════════════════════════════════════════════════════════════════════════════
# FIGURE FUNCTIONS
# ════════════════════════════════════════════════════════════════════════════════

def fig_oligo_barplot(lct_deg, dorsal_deg, ventral_deg):
    """
    Figure 21 — Bar plot: mean log2FC per oligodendrocyte category per region.
    Three bars per category (LCT, Dorsal, Ventral).
    """
    print("\n── Figure 21: Oligodendrocyte bar plot ──────────────────────────")

    regions = {
        'LCT':     (lct_deg,     COLOR_LCT),
        'Dorsal':  (dorsal_deg,  COLOR_DORSAL),
        'Ventral': (ventral_deg, COLOR_VENTRAL),
    }

    categories = list(OLIGO_CATEGORIES.keys())
    x      = np.arange(len(categories))
    width  = 0.25
    fig, ax = plt.subplots(figsize=(12, 6))

    for ri, (region_name, (deg_df, color)) in enumerate(regions.items()):
        summary = compute_region_summary(deg_df, region_name)
        means   = [summary.get(cat, {}).get('mean_lfc', 0) for cat in categories]
        sems    = [summary.get(cat, {}).get('sem', 0)      for cat in categories]
        p_vals  = [summary.get(cat, {}).get('p', 1)        for cat in categories]

        offset = (ri - 1) * width
        bars   = ax.bar(x + offset, means, width, color=color, alpha=0.85,
                        label=region_name, yerr=sems, capsize=4,
                        error_kw={'linewidth': 1})

        for i, (mean, p) in enumerate(zip(means, p_vals)):
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
            if sig:
                y_pos = (mean + sems[i] + 0.04) if mean >= 0 else (mean - sems[i] - 0.06)
                va    = 'bottom' if mean >= 0 else 'top'
                ax.text(x[i] + offset, y_pos, sig, ha='center', va=va,
                        fontsize=11, fontweight='bold')

    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xticks(x)
    ax.set_xticklabels(categories, fontsize=10)
    ax.set_ylabel('Average log2FC (ALS vs Control)', fontsize=11)
    ax.set_title('Oligodendrocyte dysregulation in ALS\nper white matter region (WM filtered)',
                 fontsize=13, fontweight='bold')

    region_patches = [mpatches.Patch(color=COLOR_LCT,     label='LCT'),
                      mpatches.Patch(color=COLOR_DORSAL,  label='Dorsal'),
                      mpatches.Patch(color=COLOR_VENTRAL, label='Ventral')]
    ax.legend(handles=region_patches, fontsize=10)

    sig_text = "* p < 0.05     ** p < 0.01     *** p < 0.001"
    fig.text(0.5, -0.04, sig_text, ha='center', fontsize=9, style='italic')
    plt.tight_layout()
    save(fig, 'Figure21_oligo_barplot.png')


def fig_oligo_heatmap(lct_deg, dorsal_deg, ventral_deg):
    """
    Figure 22 — Heatmap: individual gene log2FC per region.
    Genes grouped by oligodendrocyte category with a clean legend.
    """
    print("\n── Figure 22: Oligodendrocyte heatmap ───────────────────────────")

    gene_order = MATURE_OLIGO + OPC_MARKERS + CHOL_SYNTH
    regions    = {'LCT': lct_deg, 'Dorsal': dorsal_deg, 'Ventral': ventral_deg}

    mat = pd.DataFrame(index=gene_order, columns=regions.keys())
    for region_name, deg_df in regions.items():
        for gene in gene_order:
            if gene in deg_df.index:
                mat.loc[gene, region_name] = deg_df.loc[gene, 'log2FoldChange']
            else:
                mat.loc[gene, region_name] = np.nan
    mat = mat.astype(float)


    fig, ax = plt.subplots(figsize=(5.5, len(gene_order) * 0.25 + 2))
    vmax = max(abs(mat.values[~np.isnan(mat.values)]).max(), 0.5)
    cmap_custom = LinearSegmentedColormap.from_list(
        'als_custom',
        ['#588157', '#F5F0F0', '#C2527A'],
        N=256
    )

    im = ax.imshow(mat.values, cmap=cmap_custom, aspect='auto',
                   vmin=-vmax, vmax=vmax)


    ax.set_xticks(range(len(regions)))
    ax.set_xticklabels(list(regions.keys()), fontsize=8.5, fontweight='bold')
    
    ax.get_xticklabels()[0].set_color(COLOR_LCT)
    ax.get_xticklabels()[1].set_color(COLOR_DORSAL)
    ax.get_xticklabels()[2].set_color(COLOR_VENTRAL)

    
    ax.set_yticks(range(len(gene_order)))
    ax.set_yticklabels(gene_order, fontsize=6.5)

    
    for ti, gene in enumerate(gene_order):
        cat   = next((c for c, gs in OLIGO_CATEGORIES.items() if gene in gs), None)
        color = OLIGO_CAT_COLOR.get(cat, 'black')
        ax.get_yticklabels()[ti].set_color(color)


    for i, gene in enumerate(gene_order):
        for j, region in enumerate(regions.keys()):
            val = mat.loc[gene, region]
            if not np.isnan(val):
                ax.text(j, i, f'{val:.2f}', ha='center', va='center',
                        fontsize=6.5, color='white' if abs(val) > vmax*0.5 else 'black',
                        fontweight='bold')

  
    sep_after = [len(MATURE_OLIGO) - 0.5, len(MATURE_OLIGO) + len(OPC_MARKERS) - 0.5]
    for sep in sep_after:
        ax.axhline(sep, color='white', lw=2)

    cbar = plt.colorbar(im, ax=ax, shrink=0.5, pad=0.08)
    cbar.set_label('log2FC (ALS vs Control)', fontsize=8.5)
    cbar.ax.tick_params(labelsize=7.5)

    legend_patches = [
        mpatches.Patch(color=COLOR_AXONAL, label='Mature oligodendrocytes'),
        mpatches.Patch(color=COLOR_MICROGLIA, label='OPC markers'),
        mpatches.Patch(color=COLOR_LIPID, label='Oligo cholesterol synthesis')
    ]
    
    ax.legend(handles=legend_patches, 
              bbox_to_anchor=(1.12, 0.0),  
              loc= 'lower left', 
              fontsize=7.5, 
              frameon=False,               
              title='Gene Categories',    
              title_fontsize=8)
    ax.get_legend().get_title().set_weight('bold')
    ax.set_title('Oligodendrocyte gene expression per region\n(WM filtered)',
                 fontsize=11, fontweight='bold')
    
    plt.tight_layout()
    save(fig, 'Figure22_oligo_heatmap.png')

# ════════════════════════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════════════════════════

if __name__ == '__main__':
    print("=" * 65)
    print("  Script 8 — Oligodendrocyte Marker Analysis")
    print("=" * 65)

    # ── Step 1: Load or compute DEG results per region ────────────────────────
    lct_deg     = pd.read_csv(BASE_PATH + 'LCT_WM_filtered_DEG.csv', index_col=0) \
                  if os.path.exists(BASE_PATH + 'LCT_WM_filtered_DEG.csv') \
                  else pd.read_csv(BASE_PATH + 'DEG_results_full.csv', index_col=0)

    dorsal_deg  = load_and_run_deg(
        HEALTHY_DORSAL, ALS_DORSAL,
        DORSAL_PATH + 'dorsal_DEG_results.csv', 'Dorsal')

    ventral_deg = load_and_run_deg(
        HEALTHY_VENTRAL, ALS_VENTRAL,
        VENTRAL_PATH + 'ventral_DEG_results.csv', 'Ventral')

    # ── Step 2: Generate figures ──────────────────────────────────────────────
    print("\n── Generating figures ───────────────────────────────────────────")
    fig_oligo_barplot(lct_deg, dorsal_deg, ventral_deg)
    fig_oligo_heatmap(lct_deg, dorsal_deg, ventral_deg)

    print("\n" + "=" * 65)
    print(f"  Done! All figures saved to:")
    print(f"  {OUTPUT_PATH}")
    print("=" * 65)

  Script 8 — Oligodendrocyte Marker Analysis
  Dorsal: loading existing DEG results
  Ventral: loading existing DEG results

── Generating figures ───────────────────────────────────────────

── Figure 21: Oligodendrocyte bar plot ──────────────────────────
  Saved: Figure21_oligo_barplot.png

── Figure 22: Oligodendrocyte heatmap ───────────────────────────
  Saved: Figure22_oligo_heatmap.png

  Done! All figures saved to:
  /Users/annemijnrieff/Documents/Internship-data/Programming-data/Images-data/Region_clusters/Final_figures/Oligodendorcytes/
